In [4]:
from time import perf_counter
import numpy as np
from numpy import linalg as la
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score
from joblib import Parallel, delayed
import os

import itertools

import src.utils as utils
from src.model import Nonneg_dyn_dag_learning
from src.baselines import Dynotears


PATH = './results/tuning/'
PATH_SACHS = './datasets/sachs/'
SAVE = True 
SEED = 10
N_CPUS = os.cpu_count()
np.random.seed(SEED)

DATASET = "SYNTH"  # SYNTH, SACHS

In [ ]:
def cartesian_product(hyperparams):
    """
    Generate all combinations of hyperparameters.
    """
    param_names = list(hyperparams.keys())
    param_values = list(hyperparams.values())    
    param_combinations = [dict(zip(param_names, values)) for values in itertools.product(*param_values)]
    
    return param_combinations

args2str = lambda arguments: ''.join([f'{key[:3]}={val} ' for key, val in arguments.items()])

def print_best(key, metrics, args_combs, agg_funct='mean', all_best=False):
    agg_metric = {key: getattr(np, agg_funct)(value, axis=0) for key, value in metrics.items()}
        
    best_value = np.min(agg_metric[key])
    best_idxs = np.where(agg_metric[key] == best_value)[0]

    if not all_best:
        best_idxs = [best_idxs[0]]

    print(f'Combination{"s" if all_best else ""} with best {key} (agg: {agg_funct}):')
    for idx in best_idxs:
        print(args_combs[idx])
        print(f'shd: {agg_metric["shd"][idx]:.2f} | err W: {agg_metric["err_W"][idx]:.4f} | err A: {agg_metric["err_A"][idx]:.4f}  |' +
          f'acyc: {agg_metric["acyc"][idx]:.6f} | time: {agg_metric["time"][idx]:.2f}')

def run_grid_search_tuning(g, data_p, model_args, args_combs, model_const, std_x, fix_lamb, thr, verb=False):
    # Create data
    if DATASET == "SACHS":
        W_true = np.load(PATH_SACHS + "sachs_A_matrix.npy")
        X = np.load(PATH_SACHS + "sachs_X.npy")
    else:
        W_true, _, A_true, _, X, Y = utils.simulate_svar(**data_p)
        
    # X = utils.standarize(X) if std_x else X
    norm_W_true = np.linalg.norm(W_true)
    W_true_bin = utils.to_bin(W_true, thr)
    norm_A_true = np.linalg.norm(A_true)
    
    if verb and (g % N_CPUS == 0):
        print(f'Graph {g+1}:')

    N = data_p['n_nodes']
    n_samples = data_p['n_samples']
    n_lags = data_p['n_lags']

    shd, err_W, err_A, acyc, runtime = [np.zeros(len(args_combs))  for _ in range(5)]
    for i, args in enumerate(args_combs):

        args_aux = args.copy()
        if not fix_lamb:
            args_aux['lamb_W'] = utils.get_lamb_value(N, n_samples, args_aux['lamb_W'])
            args_aux['lamb_A'] = utils.get_lamb_value(N*n_lags, n_samples, args_aux['lamb_A'])

        model = model_const(**model_args)
        t_init = perf_counter()
        W_est, A_est = model.fit(X, Y, **args)
        t_end = perf_counter()
    
        W_est_bin = utils.to_bin(W_est, thr)

        shd[i], _, _ = utils.count_accuracy(W_true_bin, W_est_bin)
        err_W[i] = utils.compute_norm_sq_err(W_true, W_est, norm_W_true)
        err_A[i] = utils.compute_norm_sq_err(A_true, A_est, norm_A_true)
        acyc[i] = model.dagness(W_est)
        runtime[i] = t_end - t_init

        if verb:
            text = args2str(args)
            print(f'\t- {text}: shd W {shd[i]}  -  err W: {err_W[i]:.3f}  -  err A: {err_A[i]:.3f} -  acyc: {acyc[i]:.5g}  -  time: {runtime[i]:.3f}')
    
    return shd, err_W, err_A, acyc, runtime

## Experiment parameters

In [ ]:
model_const = Nonneg_dyn_dag_learning

model_args = {
    'primal_opt': 'fista',  # 'adam', 'fista', 'pgd
    'acyclicity': 'logdet',
    'restart': True,  # Only used in FISTA
}

verb = True
thr = .075
n_dags = 30 if DATASET != "SACHS" else 1
std_x = False
N = 50  
fix_lamb = False
data_params = {
    'n_nodes': N,
    'dag_graph_type': 'er',
    'dag_edges': 4*N,
    'dag_w_range': (.1, .5),
    'n_lags': 2,
    'lag_graph_type': 'er',
    'er_edges': N,
    'lag_w_range': (.1, .4),
    'exp_decay': 1.5,
    'n_samples': 1000, # 1000,
    'noise_type': 'normal',
    'var': 1
}

Hyperparams = {
    'stepsize': [1e-4, 5e-4, 3e-4, 1e-3],
    'alpha_0': [1],
    'rho_0': [.05],
    'beta': [5],
    's': [1],
    'lamb_W': [.01],
    'lamb_A': [1],
    'iters_in': [5000, 10000],
    'iters_out': [5, 10, 20],
    'tol': [1e-6],
}

print('CPUs employed:', N_CPUS)
print('Looking hyperparameters for dataset', DATASET)
# Get combination of hyperparams for grid search
args_combs = cartesian_product(Hyperparams)    

t_init = perf_counter()
results = Parallel(n_jobs=N_CPUS)(delayed(run_grid_search_tuning)
                  (g, data_params, model_args, args_combs, model_const, std_x, fix_lamb, thr, verb) for g in range(n_dags))
t_end = perf_counter()

shd, err_W, err_A, acyc, runtime = zip(*results)
metrics = {'shd': shd, 'err_W': err_W, 'err_A': err_A, 'acyc': acyc, 'time': runtime}

CPUs employed: 168
Looking hyperparameters for dataset SYNTH
Graph 1:
	- ste=0.0001 alp=1 rho=0.05 bet=5 s=1 lam=0.01 lam=1 ite=10000 ite=10 tol=1e-06 : shd W 45.0  -  err W: 0.083  -  err A: 1.594 -  acyc: 0.00022523  -  time: 6.127
	- ste=0.0001 alp=1 rho=0.05 bet=5 s=1 lam=0.01 lam=1 ite=10000 ite=10 tol=1e-06 : shd W 29.0  -  err W: 0.070  -  err A: 1.298 -  acyc: 0.00021428  -  time: 6.268
	- ste=0.0001 alp=1 rho=0.05 bet=5 s=1 lam=0.01 lam=1 ite=10000 ite=10 tol=1e-06 : shd W 32.0  -  err W: 0.101  -  err A: 1.519 -  acyc: 0.00072339  -  time: 7.117
	- ste=0.0001 alp=1 rho=0.05 bet=5 s=1 lam=0.01 lam=1 ite=10000 ite=10 tol=1e-06 : shd W 46.0  -  err W: 0.089  -  err A: 1.558 -  acyc: 0.00047076  -  time: 7.131
	- ste=0.0001 alp=1 rho=0.05 bet=5 s=1 lam=0.01 lam=1 ite=10000 ite=10 tol=1e-06 : shd W 38.0  -  err W: 0.104  -  err A: 1.522 -  acyc: 0.00059716  -  time: 7.331
	- ste=0.0001 alp=1 rho=0.05 bet=5 s=1 lam=0.01 lam=1 ite=10000 ite=10 tol=1e-06 : shd W 35.0  -  err W: 0.094

In [ ]:
print_best('shd', metrics, args_combs, all_best=True)
print_best('err_W', metrics, args_combs, all_best=True)
print()
print_best('shd', metrics, args_combs, agg_funct='median', all_best=True)
print_best('err_W', metrics, args_combs, agg_funct='median', all_best=True)


Combination with best shd (agg: mean):
{'stepsize': 0.0001, 'alpha_0': 1, 'rho_0': 0.05, 'beta': 5, 's': 1, 'lamb_W': 0.01, 'lamb_A': 1, 'iters_in': 10000, 'iters_out': 10, 'tol': 1e-06}
shd: 35.87 | err W: 0.0813 | err A: 1.5502  |acyc: 0.000310 | time: 8.98
Combination with best err_W (agg: mean):
{'stepsize': 0.0001, 'alpha_0': 1, 'rho_0': 0.05, 'beta': 5, 's': 1, 'lamb_W': 0.01, 'lamb_A': 1, 'iters_in': 10000, 'iters_out': 10, 'tol': 1e-06}
shd: 35.87 | err W: 0.0813 | err A: 1.5502  |acyc: 0.000310 | time: 8.98

Combination with best shd (agg: median):
{'stepsize': 0.0001, 'alpha_0': 1, 'rho_0': 0.05, 'beta': 5, 's': 1, 'lamb_W': 0.01, 'lamb_A': 1, 'iters_in': 10000, 'iters_out': 10, 'tol': 1e-06}
shd: 38.00 | err W: 0.0802 | err A: 1.5380  |acyc: 0.000235 | time: 8.88
Combination with best err_W (agg: median):
{'stepsize': 0.0003, 'alpha_0': 1, 'rho_0': 0.05, 'beta': 5, 's': 1, 'lamb_W': 0.01, 'lamb_A': 1, 'iters_in': 10000, 'iters_out': 10, 'tol': 1e-06}
shd: 38.00 | err W: 0.08